# 02. Multi-Agent Orchestration with CrewAI and Ollama

This notebook demonstrates how to:
1. Configure CrewAI agents to use the local Ollama LLM via CrewAI's `LLM` integration (supported by LiteLLM under the hood).
2. Wrap remote MCP server tools (Qdrant & OpenProject) into CrewAI compatible custom tools.
3. Set up collaborating agents (`Knowledge Analyst` and `Project Coordinator`).
4. Establish and kickoff a sequential Crew workflow to search contexts and create OpenProject tasks.

In [ ]:
import sys
from pathlib import Path


#  Locate and append project root so src.ragforge is importable

def get_project_root() -> Path:
    current = Path.cwd()
    for _ in range(5):
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    return Path.cwd()


project_root = get_project_root().resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Project root: {project_root}")

In [2]:
from crewai import LLM
from ragforge.config import OLLAMA_URL, DEFAULT_LLM_MODEL

print("Configuring CrewAI LLM for Ollama...")

# CrewAI uses LiteLLM internally. We pass model and base_url matching local Ollama
ollama_llm = LLM(
    model=f"ollama/{DEFAULT_LLM_MODEL}",
    base_url=OLLAMA_URL,
    temperature=0.0
)

print(f"LLM model ready: {ollama_llm.model}")
print(f"Ollama URL: {ollama_llm.base_url}")

Configuring CrewAI LLM for Ollama...
LLM model ready: ollama/gemma4:e4b
Ollama URL: http://localhost:11434


## 1. Wrapping MCP Tools for CrewAI

CrewAI uses custom tools that can be defined using the `@tool` decorator. We will wrap the underlying async MCP executors into synchronous functions so CrewAI can invoke them directly.

In [3]:
import asyncio
from crewai.tools import tool
from src.ragforge.agents.rag.agent import execute_mcp_tool, build_mcp_tools


# Wrap Qdrant search tool
@tool("search_documents")
def search_documents_tool(query: str) -> str:
    """Query the Qdrant vector database using a text prompt.
    Returns matched documents with metadata attributes and score metric.
    """
    # Execute async MCP tool synchronously for CrewAI tool compatibility
    loop = asyncio.get_event_loop()
    if loop.is_running():
        import nest_asyncio

        nest_asyncio.apply()
    return asyncio.run(execute_mcp_tool("search_documents", {"query": query}, "qdrant"))


# Wrap OpenProject list projects tool
@tool("get_project_list")
def get_project_list_tool() -> str:
    """Retrieve the list of active projects from OpenProject."""
    loop = asyncio.get_event_loop()
    if loop.is_running():
        import nest_asyncio

        nest_asyncio.apply()
    return asyncio.run(execute_mcp_tool("get_project_list", {}, "openproject"))


# Wrap OpenProject create task tool
@tool("create_project_task")
def create_project_task_tool(project_id: str, title: str, description: str) -> str:
    """Create a new task (work package) in an OpenProject project."""
    loop = asyncio.get_event_loop()
    if loop.is_running():
        import nest_asyncio

        nest_asyncio.apply()
    args = {
        "project_id": project_id,
        "title": title,
        "description": description,
        "task_type": "Task",
    }
    return asyncio.run(execute_mcp_tool("create_project_task", args, "openproject"))


print("CrewAI Custom Tools created from MCP servers:")
print(f" - {search_documents_tool.name}: {search_documents_tool.description[:60]}...")
print(f" - {get_project_list_tool.name}: {get_project_list_tool.description[:60]}...")
print(
    f" - {create_project_task_tool.name}: {create_project_task_tool.description[:60]}..."
)

CrewAI Custom Tools created from MCP servers:
 - search_documents: Tool Name: search_documents
Tool Arguments: {'query': {'desc...
 - get_project_list: Tool Name: get_project_list
Tool Arguments: {}
Tool Descript...
 - create_project_task: Tool Name: create_project_task
Tool Arguments: {'project_id'...


## 2. Define CrewAI Agents

We will establish two collaborating agents:
1. **Knowledge Analyst**: Responsible for querying Qdrant vectors to fetch information.
2. **Project Coordinator**: Responsible for interacting with OpenProject to list active projects and document findings as tasks.

In [4]:
from crewai import Agent

# Ingestion/Analyst Agent
analyst = Agent(
    role="Knowledge Analyst",
    goal="Query the Qdrant database to retrieve useful facts and context about the user requests.",
    backstory="You are an expert researcher. You query vector databases to find precise semantic contexts.",
    tools=[search_documents_tool],
    llm=ollama_llm,
    verbose=True,
)

# Project Manager Agent
project_coordinator = Agent(
    role="Project Coordinator",
    goal="Coordinate task creation in OpenProject based on findings from the Knowledge Analyst.",
    backstory="You are organized and structured. You keep track of project workflows and log packages.",
    tools=[get_project_list_tool, create_project_task_tool],
    llm=ollama_llm,
    verbose=True,
)

print("CrewAI Agents initialized!")

CrewAI Agents initialized!


## 3. Define Tasks and Kickoff Multi-Agent Collaboration

We'll define two tasks:
1. **Research Task**: Query Qdrant for facts on 'LangGraph'.
2. **Task Creation Task**: Retrieve project list and log a task summarizing the research findings in OpenProject Project ID 1.

In [ ]:
from crewai import Task, Crew, Process

# Task 1: Research database
research_task = Task(
    description="Query the vector database for facts regarding 'LangGraph'. Extract key architectural patterns.",
    expected_output="A bulleted summary of semantic contexts retrieved from the vector database.",
    agent=analyst,
)

# Task 2: Log tasks in OpenProject
log_task = Task(
    description="First, get the active project list from OpenProject. Then, create a new task in project ID '1' titled 'CrewAI Ingestion Verification' with a detailed description containing the research findings from the research task: {search_findings}.",
    expected_output="A confirmation showing that the task was created successfully under the verified project ID.",
    agent=project_coordinator,
)

# Assemble the Crew
crew = Crew(
    agents=[analyst, project_coordinator],
    tasks=[research_task, log_task],
    process=Process.sequential,
    verbose=True,
)

print("Crew configured! Kicking off sequential collaboration workflow...")
# Execute the crew (inputs mock any template parameters in tasks)
result = crew.kickoff(
    inputs={
        "search_findings": "LangGraph is an agentic framework featuring custom loop nodes and Human-in-the-loop (HITL) support."
    }
)

print("\n--- Workflow Results ---")
print(result)

Crew configured! Kicking off sequential collaboration workflow...


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 584682d2-177b-4d82-9849-c20511018a13                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Knowledge Analyst                                                                                       │
│                                                                                                                 │
│  Task: Query the vector database for facts regarding 'LangGraph'. Extract key architectural patterns.           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Knowledge Analyst                                                                                       │
│                                                                                                                 │
│  Thought: Thought: The user wants me to find facts and key architectural patterns about 'LangGraph' using the   │
│  provided vector database tool, `search_documents`. I must query the database with a relevant prompt and then   │
│  present the complete retrieved content as the final answer.                                                    │
│                                                                                                                 │
│  Using Tool: search_documents                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...


--- Workflow Results ---
Success: Task created in OpenProject via Temporal workflow. Workflow ID: op-create-task-d19f7362-c885-447d-a5c1-c52de5674392. Result: {'id': 38, 'status': 'New', 'subject': 'CrewAI Ingestion Verification', 'type': 'Task'}


╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 4a3ca9d0-81d6-4dac-b3f9-c5e8789beae2                                                                     │
│  Agent: Project Coordinator                                                                                     │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 584682d2-177b-4d82-9849-c20511018a13                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: Success: Task created in OpenProject via Temporal workflow. Workflow ID:                         │
│  op-create-task-d19f7362-c885-447d-a5c1-c52de5674392. Result: {'id': 38, 'status': 'New', 'subject': 'CrewAI    │
│  Ingestion Verification', 'type': 'Task'}                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

: 